# NER for Parameter Extraction: spaCy vs GLiNER

Medha uses Named Entity Recognition (NER) to extract parameters from user questions
before filling them into query templates.

This notebook compares two NER backends available in `ParameterExtractor`:

| Backend | Approach | Labels | Weight |
|---------|----------|--------|--------|
| **spaCy** | Pre-trained model | Fixed (PERSON, ORG, CARDINAL, …) → mapped to param names | ~15 MB |
| **GLiNER** | Zero-shot transformer | Arbitrary — uses `template.parameters` directly as labels | ~500 MB |

**Cascading strategy** (in order):
1. Regex patterns from `template.parameter_patterns`
2. GLiNER zero-shot NER *(if enabled)*
3. spaCy NER *(if enabled)*
4. Heuristic fallback (numbers + capitalized words)

**Requirements:**
```bash
pip install "medha-archai[nlp]"      # spaCy
python -m spacy download en_core_web_sm

pip install "medha-archai[gliner]"   # GLiNER
```

## Cell 1: Imports & Template Definitions

In [ ]:
import time
from medha.types import QueryTemplate
from medha.utils.nlp import ParameterExtractor

## Cell 2: Define Query Templates

We define three templates that cover different extraction scenarios:

- **`top_n`** — numeric + entity type (regex-friendly)
- **`by_person`** — proper name (NER-friendly)
- **`org_project`** — two domain entities (zero-shot NER shines here)

In [ ]:
top_n_template = QueryTemplate(
    intent="top_n",
    template_text="Show top {count} {entity}",
    query_template="SELECT * FROM {entity} LIMIT {count}",
    parameters=["count", "entity"],
    parameter_patterns={
        "count": r"\b(\d+)\b",
        "entity": r"\b(users|products|orders|employees)\b",
    },
)

person_template = QueryTemplate(
    intent="find_person",
    template_text="Find issues assigned to {person}",
    query_template="SELECT * FROM issues WHERE assignee = '{person}'",
    parameters=["person"],
    # No regex pattern: relies on NER or heuristics
)

org_project_template = QueryTemplate(
    intent="org_project_issues",
    template_text="Show open issues for {org} on project {project}",
    query_template=(
        "SELECT * FROM issues WHERE org = '{org}' "
        "AND project = '{project}' AND status = 'open'"
    ),
    parameters=["org", "project"],
    # No regex: org and project names are arbitrary strings — ideal for GLiNER
)

print("Templates defined:")
for t in [top_n_template, person_template, org_project_template]:
    print(f"  [{t.intent}] params={t.parameters}")

## Cell 3: Baseline — Regex + Heuristics Only

No NER enabled. The extractor uses regex patterns first, then falls back to
heuristics (numbers and capitalized words).

In [ ]:
ext_regex = ParameterExtractor(use_spacy=False, use_gliner=False)
print(f"spaCy available : {ext_regex.spacy_available}")
print(f"GLiNER available: {ext_regex.gliner_available}")

questions = [
    ("Show top 10 products", top_n_template),
    ("find issues assigned to Alice", person_template),
]

print()
for question, template in questions:
    try:
        params = ext_regex.extract(question, template)
        print(f"Q: {question!r}")
        print(f"   params  = {params}")
        print(f"   query   = {ext_regex.render_query(template, params)}")
    except Exception as e:
        print(f"Q: {question!r}")
        print(f"   ERROR   = {e}")
    print()

## Cell 4: spaCy NER

spaCy recognizes standard entity types (`PERSON`, `ORG`, `CARDINAL`) and maps them
to parameter names via a hardcoded table inside `_extract_via_spacy()`.

> If spaCy or the model is not installed you will see `spaCy not available` in the logs
> and the extractor falls back gracefully.

In [ ]:
ext_spacy = ParameterExtractor(use_spacy=True, use_gliner=False)
print(f"spaCy available : {ext_spacy.spacy_available}")
print(f"GLiNER available: {ext_spacy.gliner_available}")

questions_spacy = [
    ("find issues assigned to Alice", person_template),
    ("Show top 5 users", top_n_template),
]

print()
for question, template in questions_spacy:
    try:
        params = ext_spacy.extract(question, template)
        print(f"Q: {question!r}")
        print(f"   params  = {params}")
        print(f"   query   = {ext_spacy.render_query(template, params)}")
    except Exception as e:
        print(f"Q: {question!r}")
        print(f"   ERROR   = {e}")
    print()

### Inspect raw entities extracted by spaCy

`extract_entities()` returns the raw entity dict — useful for debugging what spaCy sees.

In [ ]:
if ext_spacy.spacy_available:
    texts = [
        "find issues assigned to Alice Johnson",
        "show top 20 orders",
        "list Acme Corp open tickets",
    ]
    for text in texts:
        entities = ext_spacy.extract_entities(text)
        print(f"text    : {text!r}")
        print(f"entities: {entities}")
        print()
else:
    print("spaCy not available — install with: pip install 'medha-archai[nlp]' && python -m spacy download en_core_web_sm")

## Cell 5: GLiNER — Zero-Shot NER

GLiNER's key advantage: it receives `template.parameters` directly as entity labels.
No mapping table is needed. It figures out what to extract from the label name itself.

For example, given `parameters=["org", "project"]`, GLiNER is called as:
```python
model.predict_entities(question, labels=["org", "project"])
```

This makes it effective for domain-specific entities that spaCy cannot recognize
without custom training.

> The first call downloads `urchade/gliner_medium-v2.1` (~500 MB) from HuggingFace.
> Subsequent runs use the local cache.
> Pass `gliner_model=` to select a lighter variant (e.g. `urchade/gliner_small-v2.1`).

In [ ]:
ext_gliner = ParameterExtractor(
    use_spacy=False,
    use_gliner=True,
    # gliner_model="urchade/gliner_small-v2.1",  # lighter alternative
)
print(f"spaCy available : {ext_gliner.spacy_available}")
print(f"GLiNER available: {ext_gliner.gliner_available}")

questions_gliner = [
    ("find issues assigned to Alice", person_template),
    ("show open issues for Acme Corp on project Apollo", org_project_template),
    ("list open tickets from NovaCo on project Mercury", org_project_template),
]

print()
for question, template in questions_gliner:
    try:
        params = ext_gliner.extract(question, template)
        print(f"Q: {question!r}")
        print(f"   params  = {params}")
        print(f"   query   = {ext_gliner.render_query(template, params)}")
    except Exception as e:
        print(f"Q: {question!r}")
        print(f"   ERROR   = {e}")
    print()

### Why GLiNER handles `org_project_template` better

spaCy would need `org` to map to its `ORG` label and `project` has no built-in spaCy label at all.
GLiNER receives `["org", "project"]` literally and knows what to extract from each span.

In [ ]:
if ext_gliner.gliner_available:
    question = "show open issues for Stellar Dynamics on project Helios"
    labels = org_project_template.parameters  # ["org", "project"]
    raw_entities = ext_gliner._gliner.predict_entities(question, labels)
    print(f"question : {question!r}")
    print(f"labels   : {labels}")
    print(f"entities : {raw_entities}")
else:
    print("GLiNER not available — install with: pip install 'medha-archai[gliner]'")

## Cell 6: Both Enabled — Cascade in Action

When both backends are enabled, the cascade is:
**Regex → GLiNER → spaCy → Heuristics**

Each step fills in only the parameters that the previous step missed.
Here we show a template where regex resolves `count` and GLiNER resolves `project`.

In [ ]:
ext_both = ParameterExtractor(use_spacy=True, use_gliner=True)
print(f"spaCy available : {ext_both.spacy_available}")
print(f"GLiNER available: {ext_both.gliner_available}")

hybrid_template = QueryTemplate(
    intent="top_n_project",
    template_text="Show top {count} issues for project {project}",
    query_template="SELECT TOP {count} * FROM issues WHERE project = '{project}'",
    parameters=["count", "project"],
    parameter_patterns={"count": r"\b(\d+)\b"},  # regex covers count, GLiNER covers project
)

question = "show top 3 issues for project Hermes"
params = ext_both.extract(question, hybrid_template)
print(f"\nQ: {question!r}")
print(f"   params = {params}")
print(f"   query  = {ext_both.render_query(hybrid_template, params)}")

## Cell 7: Performance Comparison

Latency of each extractor on the same question (averaged over multiple runs).

> spaCy is faster for single-call latency; GLiNER inference is heavier but
> eliminates the need to maintain label-mapping code.

In [ ]:
import statistics

N = 20
question = "find issues assigned to Alice"

results = {}

for label, ext in [
    ("regex+heuristics", ext_regex),
    ("spaCy", ext_spacy),
    ("GLiNER", ext_gliner),
    ("both (spaCy+GLiNER)", ext_both),
]:
    times = []
    for _ in range(N):
        t0 = time.perf_counter()
        try:
            ext.extract(question, person_template)
        except Exception:
            pass
        times.append((time.perf_counter() - t0) * 1000)
    results[label] = times
    print(f"{label:<22}  avg={statistics.mean(times):.3f}ms  "
          f"min={min(times):.3f}ms  max={max(times):.3f}ms")

## Cell 8: Choosing the Right Backend

| Scenario | Recommendation |
|----------|----------------|
| Params are numeric or match known patterns | Regex only (`use_spacy=False, use_gliner=False`) |
| Params are standard entities (person, org, number) | spaCy (`use_spacy=True`) |
| Params are domain-specific or unpredictable label names | GLiNER (`use_gliner=True`) |
| Mixed templates in the same app | Both enabled — cascade handles it automatically |
| Edge / resource-constrained deployment | Regex + heuristics only |

### Custom GLiNER model

You can swap the model by passing `gliner_model=` at construction time:

```python
# Lighter model (~250 MB), slightly lower accuracy
ext = ParameterExtractor(use_gliner=True, gliner_model="urchade/gliner_small-v2.1")

# Heavier model, higher accuracy on complex sentences
ext = ParameterExtractor(use_gliner=True, gliner_model="urchade/gliner_large-v2.1")
```